In [ ]:
import Pkg
Pkg.activate(".")
using NMRlab
using Plots
using JLD2
theme(:dark)
default(size=(600,400),
        tick_direction=:out,
        framestyle=:box,
        margin=(6,:mm),
        legendfontsize=8,
        guidefontsize=10,
        tickfontsize=10,
        fontfamily="Helvetica", 
        fmt=:svg,dpi=100)


In [ ]:
@load "processedData.jld2" processedData
@load "partial_spectra.jld2"

## Discussion

The above method to determine the amounts based on the partial spectra, being based on the Moore-Penrose pseudo-inverse, corresponds
to a least-squares fit of the experimental spectrum by a linear superposition of the partial spectra. This approach is simple and easy to
understand, but it suffers from a range of practical problems:

1. **Spectral overlap**: some of the peaks of the partial spectra overlap with each other
2. **Line shape**: the line width and shape of the partial spectra is artificially set by their processing
3. **Completeness**: the experimental spectrum contains many features that are not related to the partial spectra considered.
4. **Alignment**: the positions of some of the peaks may be slightly off, due to pH or solvent effects.

These shortcomings mean that the results depend on the list of metabolites that is used for analysis. Moreover, small
shifts in peak positions and spectral alignment lead to large changes in the resulting amounts. To use the above approach,
it is therefore very important to ensure good alignment, as well as narrowly defined conditions of measurement in terms
of pH, solvent, and temperature. This can be a problem, in particular when metabolites are involved that tend to 
shift the pH, e.g., lactic acid.

A possible alternative approach to least-squares minimization has been suggested by Domzal, Kazimierczuk et al. [*Anal. Chem.* 2024, **96**, 1, 188–196](https://doi.org/10.1021/acs.analchem.3c03594). Their
idea is to view the real part of the NMR spectrum as a *distribution* of intensity over the frequency domain. There *Wasserstein difference* $W_1(p,q)$ between
two distributions $p$ and $q$ corresponds to the minimial amount of "work" that needs to be done to transfer the contents of one distribution into the other. This metric
is different from the one we have used before in an important way. Let us assume that the spectra $p$ and $q$ both contain the same peak, but at different locations. The
least-squares difference we have used before is the same reqardless of how far the peak locations are from each other. The Wasserstein metric, by contrast, penalizes
larger deviations in position, and is more tolerant for small peak shifts. 

The Wasserstein metric is easy to understand if we view the two distributions as "piles" of dirt – it represents the minimum amount of "shovelling" that needs
to be carried out to convert one of them into the other. It is actually interesting that the mathematics of this problem go back to the [French mathematician Gaspard Monge](https://en.wikisource.org/wiki/1911_Encyclopædia_Britannica/Monge,_Gaspard), who
considered the construction and modification of military fortifications consisting of ditches and earthwalls. 

### Masking

One problem is that our list of partial spectra is not complete: there will always be signals in the experimental spectra from compounds that 
are not in the list. We define a mask that only lets pass parts of the spectrum that contain peaks that appear in the partial spectra.

In [ ]:
maskSpects = partialSpects .|> real .> 100.0
mask=SpectData(reduce(|, maskSpects.dat, dims=2),(NMRlab.coords(maskSpects,1),["Mask"]))

In [ ]:
a=plot()  # create an empty plot object to hold all spectra
for (k,(name, spect)) in enumerate(partialSpectra)
    plot!(a,NMRlab.coords(spect,1)./600 .+ 4.78, real.(spect[:,1]).+(k-1)*4e2,
    xaxis=:flip, ylims=5.0e3.*[-0.1,1.0], xlims=(-0.1,8.0), fmt=:svg, label=name,legend=:topleft)
end
Plots.plot!(a, NMRlab.coords(mask,1)./600 .+ 4.78,  1e5*mask,fill=true, opacity=0.35, fmt=:svg)
a

In [ ]:
a=plot(NMRlab.coords(processedData,1)./600 .+ 4.78, processedData[:,3].*mask[:,1], 
     xaxis=:flip, fmt=:svg,label="masked measured spectrum",legend=:topleft, xlims=(-0.1,8.1), ylims=1e4 .* (-0.1,1.0) )
Plots.plot!(a, NMRlab.coords(mask,1)./600 .+ 4.78,  1e4*mask,fill=true, label="mask", opacity=0.15, fmt=:svg)

The masked spectrum can now be used to compute the integral. If we interpret the spectrum as a distribution, this is equivalent to the
cumulative distribution function (CDF).

In [ ]:
a=plot(NMRlab.coords(processedData,1)./600 .+ 4.78, processedData[:,3].*mask[:,1] |> Integral(1), 
     xaxis=:flip, fmt=:svg,label="masked measured spectrum",legend=:topleft, xlims=(-0.1,8.0), ylims=10e5 .* (-0.1,1.0) )

The same can now be done for the partial spectra:

In [ ]:
partialCDF = partialSpects |> real |> Integral(1) 
a=plot()  # create an empty plot object to hold all spectra
for k=1:size(partialCDF,2)
    plot!(a,NMRlab.coords(partialCDF[:,k],1)./600 .+ 4.78, real.(partialCDF[:,k]),
    xaxis=:flip, ylims=2.0e4.*[-0.1,1.0], xlims=(-0.1,8.2), fmt=:svg, label=NMRlab.coords(partialCDF,2)[k],legend=:none)
end
a

These partial CDF are then used to construct a linear superposition, just as we have done before for the
partial spectra. We obtain a CDF version of the $B$-matrix. At the same time, we use Tikhonov regularisation
to bias the solution towards small amounts:

In [ ]:
import LinearAlgebra
F=LinearAlgebra.svd(partialCDF.dat)
λ=F.S[1]*F.S[end] # a small regularization parameter to stabilize the inversion. The choice of λ can be guided by the spectrum of singular values, or by cross-validation.
Bcdf =  SpectData(partialCDF |> x->(x'*x+λ*LinearAlgebra.I) \ x',(NMRlab.coords(partialCDF,2), NMRlab.coords(partialCDF,1)))

In [ ]:
processedCDF =  processedData .* mask |> real |> Integral(1) 
Plots.plot(NMRlab.coords(processedCDF,1)./600 .+ 4.78, processedCDF[:,:], xaxis=:flip, fmt=:svg,
    label="masked measured spectrum (CDF)",legend=false, xlims=(-0.1,8.0),  )

In [ ]:
rawCDF = SpectData(Bcdf * processedCDF , (NMRlab.coords(Bcdf,1), NMRlab.coords(processedCDF,2)))

In [ ]:
Plots.bar(
     NMRlab.coords(rawCDF,1), 
     (rawCDF[:,1],rawCDF[:,1]), 
     xticks=(0.5:39, NMRlab.coords(rawCDF,1)),
     ylabel="relative amount (a.u.)", title=NMRlab.coords(rawCDF,2)[1],xrotation=60,)

In [ ]:
Plots.plot(NMRlab.coords(processedCDF,1)./600 .+ 4.78, real(processedData[:,1]), xaxis=:flip, fmt=:svg,
    label="measured spectrum (CDF)",legend=false, xlims=(-0.1,8.0), ylims=5e5 .* (-0.1,1.0),opacity=0.5 )

Plots.plot!(NMRlab.coords(processedCDF,1)./600 .+ 4.78, 
    3000 .+ real(partialSpects) * rawCDF[:,1], xaxis=:flip,
    label="reconstructed spectrum (CDF)",legend=:topleft, xlims=(-0.1,9.1), ylims=8e3 .* (-0.1,1.0) )


In [ ]:
import NonNegLeastSquares

n_samples = size(processedData,2)
n_met = size(partialCDF,2)

rawamounts_nnls = SpectData(zeros(n_met, n_samples), (NMRlab.coords(partialCDF,2), NMRlab.coords(processedData,2)))
A = real.(partialCDF.dat)
for i in 1:n_samples
    b=real.(processedCDF.dat[:,i].*mask)
    rawamounts_nnls[:,i] = NonNegLeastSquares.nnls(A, b)
end
rawamounts_nnls 

In [ ]:
Plots.bar(
     NMRlab.coords(rawamounts_nnls,1), 
     (rawamounts_nnls[:,1],rawamounts_nnls[:,3]), 
     xticks=(0.5:39, NMRlab.coords(rawamounts_nnls,1)),
     ylabel="relative amount (a.u.)", title=NMRlab.coords(rawamounts_nnls,2)[1],xrotation=60,
     size=(800,400))

In [ ]:
Plots.plot(NMRlab.coords(processedCDF,1)./600 .+ 4.78, real(processedData[:,1]), xaxis=:flip, fmt=:svg,
    label="measured spectrum (CDF)",legend=false, xlims=(-0.1,8.0), ylims=5e5 .* (-0.1,1.0) )

Plots.plot!(NMRlab.coords(processedCDF,1)./600 .+ 4.78, 3000 .+ real(partialSpects) * rawamounts_nnls[:,1], xaxis=:flip, fmt=:svg,
    label="reconstructed spectrum (CDF)",legend=:topleft, xlims=(2.5,3.0), ylims=8e3 .* (-0.1,1.0) )
